# Discovery dataset - Schaefer100 Hurst dataframe

In [1]:
from pathlib import Path
import sys
base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)
sys.path.insert(0, str(Path(base_path) / 'CODE'))
from utils.helper_functions import remove_outliers_IQR_from_df

In [2]:
import numpy as np
import os
import scipy.io as sio
import pandas as pd

In [3]:
num_parcels = 100
field = "Hursts"
data_folder = f"{base_path}/RESULTS/discovery_dataset/hurst/raw/parcelwise/FIN_D4_modwt_1_5_padding_0_decimate_0"
atlas_info_path = f"{base_path}/DATA/templates/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.Centroid_RAS.csv"
raw_output_path = f"{base_path}/RESULTS/discovery_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest.csv"
cleaned_output_path = f"{base_path}/RESULTS/discovery_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"

## 1. Create and save dataframe

In [4]:
import os
import glob
import re
import numpy as np
import pandas as pd
import scipy.io as sio

files = sorted(glob.glob(os.path.join(data_folder, "sub*.mat")))

num_sub = len(files)
sub_names = []
data = np.full((num_sub, num_parcels), np.nan)

print(f"Found {num_sub} subject files.")

for i, filename in enumerate(files):

    base = os.path.basename(filename)
    m = re.search(r"(sub-\d+)", base)

    if m is None:
        print(f"[SKIP] Could not extract subject ID from: {base}")
        continue

    sub_name = m.group(1)   # e.g. "sub-01187"
    sub_names.append(sub_name)

    if not os.path.exists(filename):
        print(f"[SKIP] path {filename} does not exist")
        continue

    mat_data = sio.loadmat(filename)

    if field not in mat_data:
        print(f"[SKIP] field '{field}' not found in {filename}")
        print(f"       available keys: {list(mat_data.keys())}")
        continue

    hursts = np.asarray(mat_data[field]).squeeze().ravel()

    if hursts.size != num_parcels:
        print(
            f"[WARNING] {sub_name}: expected {num_parcels} parcels, "
            f"got {hursts.size}. Skipping."
        )
        continue

    data[i, :] = hursts

# If some files were skipped before appending subject names, align safely
valid_rows = ~np.all(np.isnan(data), axis=1)
data = data[valid_rows, :]

# Rebuild subject names from valid files to keep alignment
sub_names_valid = []
for filename in files:
    base = os.path.basename(filename)
    m = re.search(r"(sub-\d+)", base)
    if m is not None:
        mat_data = sio.loadmat(filename)
        if field in mat_data:
            hursts = np.asarray(mat_data[field]).squeeze().ravel()
            if hursts.size == num_parcels:
                sub_names_valid.append(m.group(1))

sub_names = sub_names_valid

print(f"Valid subjects: {len(sub_names)}")

parcel_info = pd.read_csv(atlas_info_path)

networks = parcel_info["ROI Name"].str.extract(
    r"(?:7Networks_[LR]H_)?([^_]+)"
)[0]

if len(networks) != num_parcels:
    raise ValueError(
        f"Expected {num_parcels} network labels, got {len(networks)}"
    )

# Create long DataFrame
sub_idx, parcel_idx = np.meshgrid(
    np.array(sub_names),
    np.arange(1, num_parcels + 1),
    indexing="ij"
)

df = pd.DataFrame({
    field: data.ravel(),
    "Subject": sub_idx.ravel(),          # e.g. sub-01187
    "Parcel": parcel_idx.ravel(),
    "Network": np.tile(networks.values, len(sub_names)),
})

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# Save DataFrame
os.makedirs(os.path.dirname(raw_output_path), exist_ok=True)
df.to_csv(raw_output_path, index=False)

print(f"Saved to: {raw_output_path}")
print(df.head())

Found 164 subject files.
Valid subjects: 164
Saved to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/discovery_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest.csv
     Hursts    Subject  Parcel Network
0  0.937274  sub-00395       1     Vis
1  1.132754  sub-00395       2     Vis
2  1.399384  sub-00395       3     Vis
3  1.066653  sub-00395       4     Vis
4  1.340076  sub-00395       5     Vis


## 2. Remove outliers

In [5]:
# Load the DataFrame and prepare for analysis
hurst_df = pd.read_csv(raw_output_path)

parcels = np.unique(hurst_df['Parcel'])
networks = np.unique(hurst_df['Network'])

# Clean the DataFrame by removing outliers and save the cleaned version
cleaned_hurst_df = remove_outliers_IQR_from_df(hurst_df, threshold=1.5, value_col=field)
cleaned_hurst_df.to_csv(cleaned_output_path, index=False)